In [1]:
# -------------------------------
# 1. Start Spark Session
# -------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

spark = SparkSession.builder \
    .appName("HealthcareDataOptimization") \
    .getOrCreate()


# -------------------------------
# 2. Create Sample Data
# -------------------------------
data = [
    ("P1", 80, 95, 120),
    ("P2", 130, 85, 140),
    ("P3", 75, 92, 110)
]

columns = ["patient_id", "heart_rate", "oxygen_level", "blood_pressure"]


# -------------------------------
# 3. Create DataFrame
# -------------------------------
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()


# -------------------------------
# 4. Double the Data (Main Logic)
# -------------------------------
doubled_df = df.union(df)

print("Doubled Data:")
doubled_df.show()


# -------------------------------
# 5. Add Patient Status
# -------------------------------
processed_df = doubled_df.withColumn(
    "status",
    when(col("oxygen_level") < 90, "CRITICAL")
    .when(col("heart_rate") > 120, "ALERT")
    .otherwise("NORMAL")
)

print("Processed Data with Status:")
processed_df.show()


# -------------------------------
# 6. Aggregation (Dashboard View)
# -------------------------------
summary = processed_df.groupBy("status").count()

print("Summary (for Dashboard):")
summary.show()


# -------------------------------
# 7. Save Output (Optional)
# -------------------------------
processed_df.write.mode("overwrite").csv("output_healthcare_data")

print("Data saved successfully!")

Original Data:
+----------+----------+------------+--------------+
|patient_id|heart_rate|oxygen_level|blood_pressure|
+----------+----------+------------+--------------+
|        P1|        80|          95|           120|
|        P2|       130|          85|           140|
|        P3|        75|          92|           110|
+----------+----------+------------+--------------+

Doubled Data:
+----------+----------+------------+--------------+
|patient_id|heart_rate|oxygen_level|blood_pressure|
+----------+----------+------------+--------------+
|        P1|        80|          95|           120|
|        P2|       130|          85|           140|
|        P3|        75|          92|           110|
|        P1|        80|          95|           120|
|        P2|       130|          85|           140|
|        P3|        75|          92|           110|
+----------+----------+------------+--------------+

Processed Data with Status:
+----------+----------+------------+--------------+------